In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
print(f"User Agent configurato: {os.getenv('SEC_USER_AGENT')[:15]}...") # Verifica veloce

User Agent configurato: Antonio Martell...


In [3]:
from sec_edgar_downloader import Downloader

# Inizializza
dl = Downloader("MyCompany", os.getenv("SEC_USER_AGENT"), "../data/raw")

# Scarica un 10-K di prova (es. Tesla o Microsoft)
ticker = "TSLA"
dl.get("10-K", ticker, limit=1, after="2023-01-01")

print(f"Controlla la cartella data/raw/sec-edgar-filings/{ticker}!")

Controlla la cartella data/raw/sec-edgar-filings/TSLA!


In [ ]:
import os

base_path = "../data/raw/sec-edgar-filings/TSLA/10-K"
# Prendi la prima sottocartella (quella col numero di serie)
submission_dir = os.listdir(base_path)[0]
file_path = os.path.join(base_path, submission_dir, "full-submission.txt")

with open(file_path, "r", encoding="utf-8") as f:
    raw_content = f.read()

print(f"Lunghezza totale del file: {len(raw_content)} caratteri")
print("Anteprima del contenuto:\n", raw_content[:500])

Contenuto della cartella ../data/raw/sec-edgar-filings/TSLA/10-K: ['0001628280-26-003952']
Lunghezza totale del file: 15755490 caratteri
Anteprima del contenuto:
 <SEC-DOCUMENT>0001628280-26-003952.txt : 20260129
<SEC-HEADER>0001628280-26-003952.hdr.sgml : 20260129
<ACCEPTANCE-DATETIME>20260128205503
ACCESSION NUMBER:		0001628280-26-003952
CONFORMED SUBMISSION TYPE:	10-K
PUBLIC DOCUMENT COUNT:		119
CONFORMED PERIOD OF REPORT:	20251231
FILED AS OF DATE:		20260129
DATE AS OF CHANGE:		20260128

FILER:

	COMPANY DATA:	
		COMPANY CONFORMED NAME:			Tesla, Inc.
		CENTRAL INDEX KEY:			0001318605
		STANDARD INDUSTRIAL CLASSIFICATION:	MOTOR VEHICLES & PASSENGER CAR


In [5]:
from bs4 import BeautifulSoup
import re

def clean_sec_text(raw_html):
    # Rimuovi i tag HTML
    soup = BeautifulSoup(raw_html, "lxml")
    
    # Rimuoviamo elementi inutili come script e stili
    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()

    # Estrai il testo
    text = soup.get_text(separator=" ")

    # Pulizia extra: rimuovi spazi multipli e righe vuote eccessive
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Testiamo la pulizia
clean_text = clean_sec_text(raw_content)
print(f"Lunghezza dopo la pulizia: {len(clean_text)} caratteri")
print("\nAnteprima testo pulito:\n", clean_text[5000:5500]) # Saltiamo l'header iniziale

Lunghezza dopo la pulizia: 6044600 caratteri

Anteprima testo pulito:
 us-gaap:RetainedEarningsMember 2023-12-31 0001318605 us-gaap:ParentMember 2023-12-31 0001318605 us-gaap:NoncontrollingInterestMember 2023-12-31 0001318605 srt:CumulativeEffectPeriodOfAdoptionAdjustmentMember us-gaap:RetainedEarningsMember 2023-12-31 0001318605 srt:CumulativeEffectPeriodOfAdoptionAdjustmentMember us-gaap:ParentMember 2023-12-31 0001318605 srt:CumulativeEffectPeriodOfAdoptionAdjustmentMember 2023-12-31 0001318605 us-gaap:CommonStockMember 2024-01-01 2024-12-31 0001318605 us-gaap:A


In [6]:
def refined_clean(text):
    # Rimuove stringhe tipo 'us-gaap:...' o 'srt:...' che sporcano il testo
    text = re.sub(r'\b(us-gaap|srt|dei|iso4217):\S+', '', text)
    # Rimuove date e numeri seriali isolati troppo lunghi (opzionale)
    text = re.sub(r'\b0001318605\b', '', text) 
    # Pulizia spazi bianchi
    text = re.sub(r'\s+', ' ', text).strip()
    return text

clean_text = refined_clean(clean_text)
print(f"Nuova lunghezza: {len(clean_text)} caratteri")

Nuova lunghezza: 5999372 caratteri


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuriamo lo splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # Ogni pezzo sarà di circa 1000 caratteri (~200 parole)
    chunk_overlap=200, # Sovrapposizione per non perdere il contesto tra un pezzo e l'altro
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_text(clean_text)

print(f"Finito! Abbiamo diviso il report in {len(chunks)} pezzi.")
print("\nEsempio del primo chunk:")
print(chunks[0])

Finito! Abbiamo diviso il report in 7497 pezzi.

Esempio del primo chunk:
0001628280-26-003952.txt : 20260129 0001628280-26-003952.hdr.sgml : 20260129 20260128205503 ACCESSION NUMBER: 0001628280-26-003952 CONFORMED SUBMISSION TYPE: 10-K PUBLIC DOCUMENT COUNT: 119 CONFORMED PERIOD OF REPORT: 20251231 FILED AS OF DATE: 20260129 DATE AS OF CHANGE: 20260128 FILER: COMPANY DATA: COMPANY CONFORMED NAME: Tesla, Inc. CENTRAL INDEX KEY: STANDARD INDUSTRIAL CLASSIFICATION: MOTOR VEHICLES & PASSENGER CAR BODIES [3711] ORGANIZATION NAME: 04 Manufacturing EIN: 912197729 STATE OF INCORPORATION: TX FISCAL YEAR END: 1231 FILING VALUES: FORM TYPE: 10-K SEC ACT: 1934 Act SEC FILE NUMBER: 001-34756 FILM NUMBER: 26574326 BUSINESS ADDRESS: STREET 1: 1 TESLA ROAD CITY: AUSTIN STATE: TX ZIP: 78725 BUSINESS PHONE: 512-516-8177 MAIL ADDRESS: STREET 1: 1 TESLA ROAD CITY: AUSTIN STATE: TX ZIP: 78725 FORMER COMPANY: FORMER CONFORMED NAME: TESLA MOTORS INC DATE OF NAME CHANGE: 20050222 10-K 1 tsla-20251231.htm 10-

In [9]:
import json
import os

# Definiamo il percorso di salvataggio
output_dir = "../data/chunks"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "tsla_10k_2025_chunks.json")

# Creiamo un dizionario con i metadati (utile per le fonti del RAG)
data_to_save = {
    "ticker": "TSLA",
    "report_type": "10-K",
    "period_ended": "2025-12-31",
    "chunks": chunks  # La lista di stringhe creata prima
}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data_to_save, f, ensure_ascii=False, indent=4)

print(f"✅ Successo! {len(chunks)} chunk salvati in: {output_file}")

✅ Successo! 7497 chunk salvati in: ../data/chunks/tsla_10k_2025_chunks.json
